# FB15k-237 baseline: TransE / RotatE via PyKEEN

This notebook trains classical knowledge graph embedding baselines using PyKEEN. 
Defaults are tuned for a fast preliminary run. For full runs, increase epochs to 50 and patience to 10.

**Requirements:**
`!pip install pykeen torch matplotlib`

In [ ]:
%matplotlib inline
import json
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.optim as optim

try:
    from pykeen.pipeline import pipeline
    from pykeen.datasets import FB15k237
except ImportError:
    print("Please install PyKEEN: !pip install pykeen torch", file=sys.stderr)

## Configuration
Adjust your hyperparameters here instead of using command-line arguments.

In [ ]:
MODEL = "TransE"  # Options: "TransE", "RotatE"
EPOCHS = 15       # Maximum training epochs
BATCH_SIZE = 1024
LR = 1e-3
DIM = 128         # Embedding dimension
SEED = 42
PATIENCE = 3      # Early-stopping patience
DEVICE = None     # Set to "cpu", "cuda", "mps", or None for auto-detect

# Set up output directory relative to the notebook's location
OUT_DIR = Path.cwd() / "artifacts" / "baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Utilities

In [ ]:
def best_device(target_device=None) -> str:
    """Pick the fastest available device: MPS (Apple) > CUDA > CPU."""
    if target_device:
        return target_device
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return "mps"
    return "cpu"

def dataset_stats(triples_factory) -> dict:
    """Relation-frequency and entity degree stats from a TriplesFactory."""
    triples = triples_factory.mapped_triples.numpy()  # (N, 3): h, r, t
    heads, rels, tails = triples[:, 0], triples[:, 1], triples[:, 2]

    rel_freq = Counter(rels.tolist())
    out_deg = Counter(heads.tolist())
    in_deg = Counter(tails.tolist())

    return {
        "num_triples": int(len(triples)),
        "num_entities": int(triples_factory.num_entities),
        "num_relations": int(triples_factory.num_relations),
        "top10_relations_by_freq": rel_freq.most_common(10),
        "mean_out_degree": round(sum(out_deg.values()) / max(len(out_deg), 1), 2),
        "mean_in_degree": round(sum(in_deg.values()) / max(len(in_deg), 1), 2),
        "max_out_degree": int(max(out_deg.values())) if out_deg else 0,
        "max_in_degree": int(max(in_deg.values())) if in_deg else 0,
    }

def _extract_losses(result) -> list:
    try: return list(result.losses)
    except Exception: pass
    try: return list(result.training_loop.losses_per_epoch)
    except Exception: pass
    return []

def _extract_val_mrr(result) -> list:
    try:
        history = result.stopper.results
        return [r for r in history if r is not None]
    except Exception: pass
    try: return list(result.metric_results_per_epoch)
    except Exception: pass
    return []

def save_and_show_plots(result, model_name: str, out_dir: Path) -> None:
    """Display and save training-loss and validation-MRR curves."""
    losses = _extract_losses(result)
    val_mrrs = _extract_val_mrr(result)

    if not losses and not val_mrrs:
        print("No loss/MRR history found — skipping plots.")
        return

    n_plots = int(bool(losses)) + int(bool(val_mrrs))
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 4))
    if n_plots == 1: axes = [axes]

    ax_idx = 0
    if losses:
        axes[ax_idx].plot(losses, linewidth=1.5)
        axes[ax_idx].set_title(f"{model_name} — Training loss")
        axes[ax_idx].set_xlabel("Epoch")
        axes[ax_idx].set_ylabel("Loss")
        axes[ax_idx].grid(True, linestyle="--", alpha=0.5)
        ax_idx += 1

    if val_mrrs:
        axes[ax_idx].plot(val_mrrs, linewidth=1.5, color="tab:orange")
        axes[ax_idx].set_title(f"{model_name} — Validation MRR")
        axes[ax_idx].set_xlabel("Epoch")
        axes[ax_idx].set_ylabel("Filtered MRR")
        axes[ax_idx].grid(True, linestyle="--", alpha=0.5)

    fig.tight_layout()
    plot_path = out_dir / f"{model_name}_curves.png"
    fig.savefig(plot_path, dpi=150)
    plt.show()
    print(f"Curves saved to {plot_path}")

## Execution Pipeline

In [ ]:
# Set Device
device = best_device(DEVICE)
if MODEL == "RotatE" and device == "mps":
    print("Note: MPS does not support complex ops required by RotatE — falling back to CPU.")
    device = "cpu"

# Load Dataset
print("Loading FB15k-237 ...")
dataset = FB15k237()

# Statistics
stats = dataset_stats(dataset.training)
print("\n=== Dataset statistics (training split) ===")
for k, v in stats.items():
    print(f"  {k}: {v}")

stats_path = OUT_DIR / "dataset_stats.json"
with stats_path.open("w") as f:
    json.dump(stats, f, indent=2)

# Train Model
print(
    f"\nTraining {MODEL} | d={DIM} | bs={BATCH_SIZE} | "
    f"lr={LR} | max_epochs={EPOCHS} | patience={PATIENCE} | "
    f"optimizer=adam | device={device}"
)

result = pipeline(
    model=MODEL,
    dataset=dataset,
    model_kwargs={"embedding_dim": DIM},
    training_kwargs={"num_epochs": EPOCHS, "batch_size": BATCH_SIZE},
    optimizer=optim.Adam,
    optimizer_kwargs={"lr": LR},
    stopper="early",
    stopper_kwargs={
        "frequency": 1,
        "patience": PATIENCE,
        "metric": "mean_reciprocal_rank",
        "larger_is_better": True,
    },
    random_seed=SEED,
    device=device,
)

# Save and Show Metrics / Plots
out_file = OUT_DIR / f"{MODEL}_summary.txt"
with out_file.open("w", encoding="utf-8") as f:
    header = (
        f"model={MODEL} dim={DIM} epochs={EPOCHS} "
        f"bs={BATCH_SIZE} lr={LR} patience={PATIENCE} "
        f"optimizer=adam device={device} seed={SEED}\n\n"
    )
    f.write(header)
    print("\n=== Test metrics ===")
    try:
        metrics = result.metric_results.to_flat_dict()
        for metric, val in sorted(metrics.items()):
            line = f"{metric}: {val:.4f}" if isinstance(val, float) else f"{metric}: {val}"
            print(line)
            f.write(line + "\n")
    except Exception:
        f.write(repr(result))

if hasattr(result, "save_to_directory"):
    result.save_to_directory(str(OUT_DIR / f"pykeen_{MODEL}"))

save_and_show_plots(result, MODEL, OUT_DIR)
print(f"\nSummary written to {out_file}")